# 01. GCR Baseline Reproduction

**Purpose:** Reproduce the Graph-Constrained Reasoning (GCR) baseline on WebQSP and CWQ.

**Output:** Saved predictions JSONL at `results/GenPaths/{dataset}/GCR-{model}/test/{config}/predictions.jsonl`

**Reference:** Luo et al. (2025) -- GCR achieves 92.6 Hits@1 on WebQSP, 75.8 on CWQ.

Based on: `workflow/predict_paths_and_answers.py` + `scripts/graph_constrained_decoding.sh`

In [ ]:
# @title 1. Install & Setup
import sys
import os
import torch
import json
from tqdm import tqdm
from datasets import load_dataset

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
gpu_arch = {75: "T4", 89: "L4", 80: "A100", 90: "H100"}
gpu_cc = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
has_a100 = "A100" in gpu_name or "H100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  Arch: {gpu_arch.get(gpu_cc, 'unknown')}")

!pip install -q transformers==4.44.2 "accelerate>=0.30.1" "datasets>=2.19.2" "marisa-trie>=1.2.0" "networkx>=3.0" "scikit-learn>=1.5.0" "tiktoken>=0.7.0" "sentencepiece>=0.2.0" "protobuf>=5.27.1"

flash_attn_installed = False
if has_a100:
    try:
        import flash_attn
        flash_attn_installed = True
        print(f"flash-attn already installed: {flash_attn.__version__}")
    except ImportError:
        print("Installing flash-attn (wheel)...")
        try:
            !pip install -q flash-attn --no-build-isolation
            import flash_attn
            flash_attn_installed = True
        except Exception:
            print("Wheel failed, trying source build...")
            try:
                !pip install -q flash-attn --no-build-isolation --no-cache-dir
                import flash_attn
                flash_attn_installed = True
            except Exception:
                print("flash-attn install failed, falling back to sdpa.")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca
%cd graph-constrained-reasoning
sys.path.insert(0, '.')
from src.llms import get_registed_model

In [ ]:
# @title 2. Configuration
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"  # ~16GB, needs A100
MODEL_NAME = "gcr"                                      # triggers GraphConstrainedDecodingModel

# For gated models, set your HF token:
# os.environ["HF_TOKEN"] = "hf_your_token_here"

DATA_PATH = "rmanluo"
DATASETS = ["RoG-webqsp", "RoG-cwq"]
SPLIT = "test"

INDEX_LEN = 2         # max path length
K = 5                 # beam width
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256
N_PROC = 1            # multiprocessing workers (set >1 for speed, but careful with GPU memory)

ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"

DRIVE_BASE = "/content/drive/MyDrive/gcr_results" if IN_COLAB else "results/GenPaths"
PREDICT_PATH = DRIVE_BASE
FORCE_RERUN = True          # overwrite existing results

print(f"Model: {MODEL_PATH}")
print(f"Attention: {ATTN_IMPL}")
print(f"Datasets: {DATASETS}")
print(f"Beam width k: {K}")
print(f"Max path length: {INDEX_LEN}")

In [ ]:
# @title 3. Load Model (one time, shared across datasets)
import argparse
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

try:
    print("Loading model (this takes 1-2 minutes)...")
    model = LLM(args)
    model.prepare_for_inference()
    print("Model loaded.")
except Exception as e:
    print(f"Model loading failed: {e}")
    raise

In [ ]:
# @title 4. Run GCR Baseline on Selected Dataset

def get_output_file(path, force=False):
    """Open output file for writing predictions. Resume from existing if not forced."""
    if not os.path.exists(path) or force:
        fout = open(path, "w")
        return fout, []
    else:
        with open(path, "r") as f:
            processed = [json.loads(line)["id"] for line in f]
        fout = open(path, "a")
        return fout, processed

def predict_one(data, processed_list, input_builder, model):
    """Run GCR prediction on a single question. Returns dict or None."""
    qid = data["id"]
    if qid in processed_list:
        return None
    input_query, ground_paths, trie = input_builder.process_input(data)
    if trie is None:
        return None
    start_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_START_TOKEN)
    end_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_END_TOKEN)
    llm_input = model.prepare_model_prompt(input_query)
    prediction = model.generate_sentence(
        llm_input, trie,
        start_token_ids=start_token_ids,
        end_token_ids=end_token_ids,
        enable_constrained_by_default=False,
    )
    if prediction is None:
        return None
    return {
        "id": qid,
        "question": data["question"],
        "prediction": prediction,
        "ground_truth": data["answer"],
        "ground_truth_paths": ground_paths,
        "input": input_query,
    }

SELECTED_DATASET = "RoG-webqsp"  # or "RoG-cwq"

print(f"Processing {SELECTED_DATASET}...")
dataset = load_dataset(f"{DATA_PATH}/{SELECTED_DATASET}", split=SPLIT)
print(f"Loaded {len(dataset)} questions")

input_builder = PathGenerationWithAnswerPromptBuilder(
    model.tokenizer, PROMPT_MODE, index_path_length=INDEX_LEN
)

postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}"
data_name = SELECTED_DATASET
model_short = MODEL_PATH.split("/")[-1]
output_dir = os.path.join(PREDICT_PATH, data_name, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")

fout, processed = get_output_file(os.path.join(output_dir, "predictions.jsonl"), force=FORCE_RERUN)
print(f"Already processed: {len(processed)} questions")

try:
    for data in tqdm(dataset, desc="Generating"):
        res = predict_one(data, processed, input_builder, model)
        if res is not None:
            fout.write(json.dumps(res) + "\n")
            fout.flush()
except Exception as e:
    print(f"Inference failed: {e}")
finally:
    fout.close()

from src.utils.qa_utils import eval_path_result_w_ans
print("\n=== Evaluation ===")
eval_path_result_w_ans(os.path.join(output_dir, "predictions.jsonl"))

In [ ]:
# @title 5. Compare with Published Results
print("""
=== GCR Published Results (Luo et al. 2025) ===

                    WebQSP     CWQ
  GCR (Llama-3.1-8B)
    Hits@1           92.6      75.8
    F1               89.8      69.4
    Faithfulness    100%      100%

Your reproduction should be within 1-2% of these.
""")